# Preprocessing and Feature Engineering


This notebook prepares the Microsoft (MSFT) daily OHLC data for the 90-dat direction prediction task.

This notebook does the following things:
- load the raw MSFT data
- create the 90-day directional target
- apply smoothing and technical-indicator feature engineering
- inspect missing values created by rolling features
- define time-ordered train / validation / test splits
- apply scaling 
- prepare a final feature step for model notebooks

In [6]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.preprocessing import MinMaxScaler

# to import from the src/ folder when running the notebook
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from features import full_predictors

In [7]:
TICKER = "MSFT"
START_DATE = "1999-01-01"
END_DATE = "2026-04-01"   

raw_df = yf.Ticker(TICKER).history(start=START_DATE, end=END_DATE)

raw_df = raw_df[["Open", "High", "Low", "Close", "Volume"]].copy()
raw_df = raw_df.dropna().sort_index()

print(f"Shape: {raw_df.shape}")
print(f"Date range: {raw_df.index.min().date()} to {raw_df.index.max().date()}")

raw_df.head()

Shape: (6852, 5)
Date range: 1999-01-04 to 2026-03-31


,Open,High,Low,Close,Volume
Date,,,,,
1999-01-04 00:00:00-05:00,21.271745,22.131185,21.236034,21.483629,69305200
1999-01-05 00:00:00-05:00,21.616958,22.550201,21.550297,22.321651,64281600
1999-01-06 00:00:00-05:00,22.778748,23.083481,22.359741,23.045389,69064800
1999-01-07 00:00:00-05:00,22.816845,22.950166,22.588296,22.931120,51150400
1999-01-08 00:00:00-05:00,23.188237,23.235851,22.397837,22.835890,50244800


In [8]:
df = raw_df.copy()

df["Close_t_plus_90"] = df["Close"].shift(-90)
df["Target"] = (df["Close_t_plus_90"] > df["Close"]).astype(int)

# remove rows where the future close is unavailable
df = df.dropna(subset=["Close_t_plus_90"]).copy()

print(df[["Close", "Close_t_plus_90", "Target"]].head())
print()
print("Class distribution:")
print(df["Target"].value_counts(normalize=True).sort_index())

                               Close  Close_t_plus_90  Target
Date                                                         
1999-01-04 00:00:00-05:00  21.483629        24.111948       1
1999-01-05 00:00:00-05:00  22.321651        23.426313       1
1999-01-06 00:00:00-05:00  23.045389        24.111948       1
1999-01-07 00:00:00-05:00  22.931120        23.978628       1
1999-01-08 00:00:00-05:00  22.835890        24.169088       1

Class distribution:
Target
0    0.356995
1    0.643005
Name: proportion, dtype: float64


In [9]:
# exponential smoothing used before technical-indicator generation
df["EMA_Close"] = df["Close"].ewm(alpha=0.8, adjust=False).mean()
df["EMA_Open"]  = df["Open"].ewm(alpha=0.8, adjust=False).mean()
df["EMA_High"]  = df["High"].ewm(alpha=0.8, adjust=False).mean()
df["EMA_Low"]   = df["Low"].ewm(alpha=0.8, adjust=False).mean()

df[["Close", "EMA_Close", "Open", "EMA_Open"]].head()

,Close,EMA_Close,Open,EMA_Open
Date,,,,
1999-01-04 00:00:00-05:00,21.483629,21.483629,21.271745,21.271745
1999-01-05 00:00:00-05:00,22.321651,22.154047,21.616958,21.547915
1999-01-06 00:00:00-05:00,23.045389,22.867121,22.778748,22.532582
1999-01-07 00:00:00-05:00,22.931120,22.918320,22.816845,22.759993
1999-01-08 00:00:00-05:00,22.835890,22.852376,23.188237,23.102588


In [ ]:
# function from src/features.py to generate technical indicators and other features
df = full_predictors(df)

print(f"Shape after feature engineering: {df.shape}")
print()
print("New feature columns:")
print(df.columns.tolist())

Shape after feature engineering: (6672, 27)

New feature columns:
['Open', 'High', 'Low', 'Close', 'Volume', 'Close_t_plus_90', 'Target', 'EMA_Close', 'EMA_Open', 'EMA_High', 'EMA_Low', 'stoch_k', 'stoch_d', 'Williams %R', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist', 'OBV', 'ROC (Rate of Change)', 'return_5', 'return_14', 'return_90', 'volatility', 'momentum_7', 'momentum_30', 'momentum_90']


In [11]:
print(f"Final shape after target + features: {df.shape}")
print(f"Total missing values remaining: {df.isna().sum().sum()}")

feature_preview = [
    "Close", "EMA_Close",
    "stoch_k", "stoch_d",
    "RSI", "Williams %R",
    "MACD", "MACD_signal", "MACD_hist",
    "OBV", "ROC (Rate of Change)",
    "return_5", "return_14", "return_90",
    "volatility", "momentum_7", "momentum_30", "momentum_90",
    "Target"
]

df[feature_preview].head()

Final shape after target + features: (6672, 27)
Total missing values remaining: 0


,Close,EMA_Close,stoch_k,stoch_d,RSI,Williams %R,MACD,MACD_signal,MACD_hist,OBV,ROC (Rate of Change),return_5,return_14,return_90,volatility,momentum_7,momentum_30,momentum_90,Target
Date,,,,,,,,,,,,,,,,,,,
1999-05-13 00:00:00-04:00,24.111948,24.187282,21.993567,25.960418,39.832659,78.006433,-0.587743,-0.530023,-0.057720,396922400.0,-7.359378,0.015645,-0.073594,0.125847,0.216502,1.011247,0.879485,1.125847,1
1999-05-14 00:00:00-04:00,23.426313,23.578507,5.273206,19.178914,35.330712,94.726794,-0.633033,-0.550625,-0.082408,312521400.0,-11.607929,-0.019086,-0.116079,0.064298,0.311382,0.979450,0.839205,1.064298,1
1999-05-17 00:00:00-04:00,24.111948,24.005260,19.918021,15.728265,40.414697,80.081979,-0.606608,-0.561822,-0.044786,379371600.0,-7.003207,-0.009445,-0.070032,0.049772,0.290499,1.008001,0.834569,1.049772,1
1999-05-18 00:00:00-04:00,23.978628,23.983954,25.819407,17.003545,40.244594,74.180593,-0.589627,-0.567383,-0.022244,315513200.0,-4.763316,-0.013787,-0.047633,0.046497,0.293756,0.997782,0.836149,1.046497,1
1999-05-19 00:00:00-04:00,24.169088,24.132062,34.346700,26.694709,42.069949,65.653300,-0.554410,-0.564788,0.010378,362024000.0,-3.635158,-0.014560,-0.036352,0.055998,0.289289,0.995788,0.847184,1.055998,1


In [12]:
# clean list for candidate features to use in modeling
feature_cols = [
    "Open", "High", "Low", "Close", "Volume",
    "EMA_Open", "EMA_High", "EMA_Low", "EMA_Close",
    "stoch_k", "stoch_d",
    "RSI", "Williams %R",
    "MACD", "MACD_signal", "MACD_hist",
    "OBV", "ROC (Rate of Change)",
    "return_5", "return_14", "return_90",
    "volatility",
    "momentum_7", "momentum_30", "momentum_90"
]

X = df[feature_cols].copy()
y = df["Target"].copy()

print(f"Number of predictor columns: {len(feature_cols)}")
print(feature_cols)

Number of predictor columns: 25
['Open', 'High', 'Low', 'Close', 'Volume', 'EMA_Open', 'EMA_High', 'EMA_Low', 'EMA_Close', 'stoch_k', 'stoch_d', 'RSI', 'Williams %R', 'MACD', 'MACD_signal', 'MACD_hist', 'OBV', 'ROC (Rate of Change)', 'return_5', 'return_14', 'return_90', 'volatility', 'momentum_7', 'momentum_30', 'momentum_90']


In [ ]:
# split into train/val/test sets (50%/20%/30%) while preserving temporal order
# this split is used for the CNN model in 03_cnn_model.ipynb
n = len(X)

train_end = int(0.5 * n)
val_end = int(0.7 * n)

X_train = X.iloc[:train_end].copy()
X_val = X.iloc[train_end:val_end].copy()
X_test = X.iloc[val_end:].copy()

y_train = y.iloc[:train_end].copy()
y_val = y.iloc[train_end:val_end].copy()
y_test = y.iloc[val_end:].copy()

print("Train shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_val.shape, y_val.shape)
print("Test shape:", X_test.shape, y_test.shape)

print("\nDate ranges:")
print("Train:", X_train.index.min().date(), "to", X_train.index.max().date())
print("Validation:", X_val.index.min().date(), "to", X_val.index.max().date())
print("Test:", X_test.index.min().date(), "to", X_test.index.max().date())

Train shape: (3336, 25) (3336,)
Validation shape: (1334, 25) (1334,)
Test shape: (2002, 25) (2002,)

Date ranges:
Train: 1999-05-13 to 2012-08-13
Validation: 2012-08-14 to 2017-11-30
Test: 2017-12-01 to 2025-11-18


In [15]:
# scale features using MinMaxScaler (fit on train only, then transform val/test)
scaler = MinMaxScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_val_scaled = pd.DataFrame(
    scaler.transform(X_val),
    columns=X_val.columns,
    index=X_val.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Scaled train shape:", X_train_scaled.shape)
print("Scaled validation shape:", X_val_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)

X_train_scaled.head()


Scaled train shape: (3336, 25)
Scaled validation shape: (1334, 25)
Scaled test shape: (2002, 25)


,Open,High,Low,Close,Volume,EMA_Open,EMA_High,EMA_Low,EMA_Close,stoch_k,stoch_d,RSI,Williams %R,MACD,MACD_signal,MACD_hist,OBV,ROC (Rate of Change),return_5,return_14,return_90,volatility,momentum_7,momentum_30,momentum_90
Date,,,,,,,,,,,,,,,,,,,,,,,,,
1999-05-13 00:00:00-04:00,0.542057,0.536207,0.532769,0.516068,0.070096,0.541165,0.535267,0.532008,0.520820,0.218722,0.251941,0.348548,0.781278,0.369530,0.369802,0.460847,0.326970,0.332963,0.571908,0.332963,0.526995,0.069168,0.487048,0.323319,0.526995
1999-05-14 00:00:00-04:00,0.514673,0.514195,0.502031,0.488848,0.126416,0.519993,0.519057,0.507103,0.496525,0.050852,0.182346,0.289885,0.949148,0.359311,0.364544,0.444251,0.310920,0.272084,0.500730,0.272084,0.471305,0.103750,0.430357,0.276374,0.471305
1999-05-17 00:00:00-04:00,0.496415,0.509640,0.506641,0.516068,0.096154,0.501108,0.512151,0.505824,0.513556,0.197884,0.146934,0.356132,0.802116,0.365274,0.361687,0.469541,0.323632,0.338066,0.520489,0.338066,0.458161,0.096139,0.481261,0.270971,0.458161
1999-05-18 00:00:00-04:00,0.526083,0.517989,0.518168,0.510775,0.090995,0.521137,0.517487,0.514825,0.512706,0.257133,0.160021,0.353916,0.742867,0.369105,0.360268,0.484695,0.311489,0.370162,0.511590,0.370162,0.455198,0.097326,0.463040,0.272812,0.455198
1999-05-19 00:00:00-04:00,0.526844,0.513435,0.509715,0.518337,0.061084,0.525753,0.514890,0.509837,0.518616,0.342746,0.259477,0.377701,0.657254,0.377051,0.360930,0.506625,0.320333,0.386328,0.510005,0.386328,0.463795,0.095697,0.459485,0.285673,0.463795


In [19]:
train_processed = X_train_scaled.copy()
train_processed["Target"] = y_train.values

val_processed = X_val_scaled.copy()
val_processed["Target"] = y_val.values

test_processed = X_test_scaled.copy()
test_processed["Target"] = y_test.values



In [20]:
output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

train_processed.to_csv(output_dir / "msft_train_processed.csv")
val_processed.to_csv(output_dir / "msft_val_processed.csv")
test_processed.to_csv(output_dir / "msft_test_processed.csv")

print("Saved files:")
for file in sorted(output_dir.glob("msft_*_processed.csv")):
    print("-", file.name)

Saved files:
- msft_test_processed.csv
- msft_train_processed.csv
- msft_val_processed.csv


## Outputs

This notebook produces:
- engineered MSFT features for the 90-day direction task
- chronological train / validation / test splits
- min-max scaled versions of each split
- saved processed datasets in `data/processed/`